In [6]:
import sys; sys.path.append("../src")
import pandas as pd
from preprocess import full_preprocess

raw = pd.read_csv("../data/UpdatedResumeDataSet.csv").drop_duplicates(subset="Resume").reset_index(drop=True)
proc = pd.read_csv("../data/resumes_processed.csv")
print(full_preprocess(raw["Resume"][0]) == proc["processed"][0])    # should print True

True


In [7]:
from skills import extract_skills
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

cleaned = pd.read_csv("../data/resumes_cleaned.csv")

tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
tfidf.fit(proc["processed"])

def text_similarity(resume_raw, jd_raw):
    vecs = tfidf.transform([full_preprocess(resume_raw), full_preprocess(jd_raw)])
    return float(cosine_similarity(vecs[0], vecs[1])[0][0])

def match_report(resume_raw, jd_raw):
    r_skills, j_skills = set(extract_skills(resume_raw)), set(extract_skills(jd_raw))
    overlap = r_skills & j_skills
    skill_score = len(overlap) / len(j_skills) if j_skills else 0
    text_score = text_similarity(resume_raw, jd_raw)
    return {
        "overall": round(100 * (0.4 * text_score + 0.6 * skill_score), 1),
        "text_similarity": round(100 * text_score, 1),
        "skill_match": round(100 * skill_score, 1),
        "matched_skills": sorted(overlap),
        "missing_skills": sorted(j_skills - r_skills),
    }

In [8]:
jds = {
    "DS": "We are hiring a Data Scientist with experience in Python, machine learning, deep learning, TensorFlow, SQL and data analysis. Knowledge of AWS is a plus.",
    "HR": "Looking for an HR executive skilled in recruitment, payroll and employee relations.",
    "Java": "Java developer with Spring, Hibernate, MySQL and REST API experience.",
}

for cat in ["Data Science", "HR", "Java Developer"]:
    i = cleaned.index[cleaned["Category"] == cat][0]
    for name, jd in jds.items():
        r = match_report(cleaned["clean_resume"][i], jd)
        print(f"{cat:15} vs {name:5} -> overall {r['overall']:5} | skills {r['skill_match']:5} | missing {r['missing_skills']}")
    print()

Data Science    vs DS    -> overall  37.6 | skills  57.1 | missing ['aws', 'data analysis', 'tensorflow']
Data Science    vs HR    -> overall   0.0 | skills   0.0 | missing ['employee relations', 'hr', 'payroll', 'recruitment']
Data Science    vs Java  -> overall  24.5 | skills  40.0 | missing ['hibernate', 'rest api', 'spring']

HR              vs DS    -> overall   0.0 | skills   0.0 | missing ['aws', 'data analysis', 'deep learning', 'machine learning', 'python', 'sql', 'tensorflow']
HR              vs HR    -> overall  16.7 | skills  25.0 | missing ['employee relations', 'payroll', 'recruitment']
HR              vs Java  -> overall   0.0 | skills   0.0 | missing ['hibernate', 'java', 'mysql', 'rest api', 'spring']

Java Developer  vs DS    -> overall   0.0 | skills   0.0 | missing ['aws', 'data analysis', 'deep learning', 'machine learning', 'python', 'sql', 'tensorflow']
Java Developer  vs HR    -> overall   0.0 | skills   0.0 | missing ['employee relations', 'hr', 'payroll', 'rec

In [9]:
i = cleaned.index[cleaned["Category"] == "Java Developer"][0]
print(extract_skills(cleaned["clean_resume"][i]))
print(cleaned["clean_resume"][i][:500])

['communication', 'javascript', 'java']
education details august 2010 to may 2017 be electronics communication jabalpur madhya pradesh takshshila institute of technology java developer skill details java javascript exprience 6 monthscompany details company wab it softwere pvt. ltd. description jr. java developer


In [10]:
text = cleaned["clean_resume"][i]
for kw in ["spring", "hibernate", "mysql", "rest", "sql", "servlet", "jsp"]:
    print(kw, kw in text)

spring False
hibernate False
mysql False
rest False
sql False
servlet False
jsp False


In [11]:
import pandas as pd
cleaned = pd.read_csv("../data/resumes_cleaned.csv")
i = cleaned.index[cleaned["Category"] == "Data Science"][0]
print(cleaned["clean_resume"][i])

skills programming languages python pandas numpy scipy scikit learn matplotlib sql java javascript jquery. machine learning regression svm na ve bayes knn random forest decision trees boosting techniques cluster analysis word embedding sentiment analysis natural language processing dimensionality reduction topic modelling lda nmf pca neural nets. database visualizations mysql sqlserver cassandra hbase elasticsearch d3.js dc.js plotly kibana matplotlib ggplot tableau. others regular expression html css angular 6 logstash kafka python flask git docker computer vision open cv and understanding of deep learning.education details data science assurance associate data science assurance associate ernst young llp skill details javascript exprience 24 months jquery exprience 24 months python exprience 24 monthscompany details company ernst young llp description fraud investigations and dispute services assurance technology assisted review tar technology assisted review assists in accelerating t